# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jasleen13/ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb

import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# Print real column names before trusting anything below, since I could not run this myself.
cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_march']} LIMIT 0").df()
print(cols[['column_name', 'column_type']].to_string(index=False))


             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
client_cols = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']} LIMIT 0").df()
print(client_cols[['column_name', 'column_type']].to_string(index=False))

        column_name column_type
     client_hash_id     VARCHAR
          is_active     BOOLEAN
     has_gsc_access     BOOLEAN
     has_ga4_access     BOOLEAN
     access_profile     VARCHAR
client_created_date        DATE
client_updated_date        DATE
     gsc_data_start        DATE
     ga4_data_start        DATE


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Duplicate-grain rows found: {len(grain_check)} (0 means the stated grain holds)")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate-grain rows found: 0 (0 means the stated grain holds)


,report_date,client_hash_id,content_hash_id,n


In [7]:
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_march']}
""").df()

print(counts.to_string(index=False))

 n_rows  n_content  n_clients   min_date   max_date
9841378     331437         55 2026-03-01 2026-03-31


In [8]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_not_available_rows
    FROM {TABLES['fact_march']}
""").df()

print(availability.to_string(index=False))
print()
survive_pct = availability['ga4_available_rows'][0] / availability['total_rows'][0]
print(f"Share of March rows with GA4 data actually available: {survive_pct:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  ga4_available_rows  ga4_not_available_rows
    9841378            413966.0               9427412.0

Share of March rows with GA4 data actually available: 4.2%


In [9]:
features = con.sql(f"""
    WITH monthly AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(f.gsc_impressions)                        AS imp_month,
            SUM(f.gsc_clicks)                              AS clk_month,
            AVG(NULLIF(f.gsc_avg_position, 0))             AS avg_position_month,
            COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS days_with_impressions,
            ANY_VALUE(c.gsc_data_start)                    AS client_gsc_start
        FROM {TABLES['fact_march']} f
        JOIN {TABLES['dim_clients']} c ON f.client_hash_id = c.client_hash_id
        GROUP BY 1, 2
        HAVING imp_month >= 100
    )
    SELECT *,
           clk_month / imp_month AS ctr_month,
           DATE '2026-03-01' - client_gsc_start AS client_history_days,
           CASE
               WHEN avg_position_month <= 3  THEN 'top_3'
               WHEN avg_position_month <= 10 THEN 'page_1'
               WHEN avg_position_month <= 20 THEN 'striking'
               WHEN avg_position_month <= 50 THEN 'page_3_5'
               ELSE 'deep'
           END AS position_bucket
    FROM monthly
    WHERE avg_position_month IS NOT NULL
""").df()

print(f"{len(features):,} content items with imp_month >= 100 and a real position reading")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 content items with imp_month >= 100 and a real position reading


,client_hash_id,content_hash_id,imp_month,clk_month,avg_position_month,days_with_impressions,client_gsc_start,ctr_month,client_history_days,position_bucket
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.331238,29,2025-06-07,0.000000,267,page_1
1,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.908100,31,2025-06-07,0.001112,267,page_1
2,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,30,2025-06-07,0.000000,267,page_1
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0,5.177774,31,2025-06-07,0.000000,267,page_1
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0,4.685335,31,2025-06-07,0.001295,267,page_1


In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

tier_median_ctr = features.groupby('position_bucket')['ctr_month'].transform('median')
features['is_underperforming'] = (features['ctr_month'] < tier_median_ctr).astype(int)

# WITH the leak: ctr_month is literally the column the label was thresholded on
leaky_cols = ['imp_month', 'avg_position_month', 'ctr_month', 'days_with_impressions', 'client_history_days']
honest_cols = ['imp_month', 'avg_position_month', 'days_with_impressions', 'client_history_days']

model_data = features.dropna(subset=leaky_cols + ['is_underperforming'])
X_leak = model_data[leaky_cols]
X_honest = model_data[honest_cols]
y = model_data['is_underperforming']

for name, X in [('WITH the leak (ctr_month included)', X_leak),
                ('honest (ctr_month removed)', X_honest)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    print(f"{name:38} ROC AUC: {auc:.3f}")


WITH the leak (ctr_month included)     ROC AUC: 0.803
honest (ctr_month removed)             ROC AUC: 0.709


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
onboarding_check = con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_data_start < DATE '2026-03-01' THEN 1 ELSE 0 END) AS started_before_march,
        SUM(CASE WHEN gsc_data_start >= DATE '2026-03-01' AND gsc_data_start < DATE '2026-04-01' THEN 1 ELSE 0 END) AS started_during_march,
        SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS no_start_date,
        COUNT(*) AS total_clients
    FROM {TABLES['dim_clients']}
""").df()

print(onboarding_check.to_string(index=False))

 started_before_march  started_during_march  no_start_date  total_clients
                 52.0                   5.0           37.0            104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.